# 🚢 Pierwszy notebook — Titanic

> **Zbiór danych:** `titanic.csv` — pasażerowie RMS Titanic (1912) wraz z informacją, kto przeżył  
> **Cel:** poznać podstawy `pandas` na realnych danych

---

## 📑 Spis treści

1. [Wczytanie danych](#1)
2. [Pierwszy rzut oka](#2)
3. [Typy danych](#3)
4. [Sortowanie](#4)
5. [Filtrowanie wierszy](#5)
6. [Kto przeżył?](#6)
7. [Statystyki opisowe](#7)
8. [Powrót z DataFrame do Pythona](#8)
9. [🎯 Zadania do samodzielnego rozwiązania](#9)

<a id="1"></a>
## 1. 📥 Wczytanie danych

Importujemy bibliotekę **pandas** (konwencjonalnie jako `pd`) i wczytujemy plik CSV do obiektu `DataFrame`.

In [ ]:
import pandas as pd

dataset = pd.read_csv('titanic.csv')
dataset

Nie musimy wyświetlać całej tabeli — możemy wybrać tylko interesujące nas kolumny:

- **jedna kolumna** → `dataset['Name']` (zwraca obiekt `Series`)
- **kilka kolumn** → `dataset[['Name', 'Age', 'Survived']]` (lista nazw w nawiasach kwadratowych → zwraca `DataFrame`)

In [ ]:
# Jedna kolumna
dataset['Name']

In [ ]:
# Kilka kolumn naraz
dataset[['Name', 'Age', 'Survived']]

<a id="2"></a>
## 2. 👀 Pierwszy rzut oka

Metoda `head()` pokazuje pierwsze wiersze — domyślnie **5**. Świetna do szybkiego podejrzenia struktury.

Jej odpowiednikiem jest `tail()`, który pokazuje **ostatnie** wiersze. Obie przyjmują argument z liczbą wierszy, np. `head(10)`.

In [ ]:
dataset.head()

In [ ]:
dataset.tail()

<a id="3"></a>
## 3. 🔤 Typy danych

Atrybut `dtypes` mówi, jak pandas zinterpretował każdą kolumnę:

| Typ pandas | Znaczenie |
|------------|-----------|
| `object`   | tekst / kategoria |
| `int64`    | liczba całkowita |
| `float64`  | liczba zmiennoprzecinkowa |

💡 Zwróć uwagę, że `Age` to `float64` — bo zawiera braki danych (`NaN`).

In [ ]:
dataset.dtypes

In [ ]:
# Ile braków danych (NaN) w każdej kolumnie?
dataset.isna().sum()

<a id="4"></a>
## 4. ↕️ Sortowanie

Tabelę (`DataFrame`) sortujemy metodą `sort_values()`:

- `by='Age'` — kolumna, według której sortujemy
- `ascending=True` (domyślnie) — rosnąco; `ascending=False` — malejąco
- można podać **listę kolumn** (sortowanie wielopoziomowe) oraz listę kierunków

💡 `sort_values()` zwraca **nowy** DataFrame — oryginał zostaje bez zmian (chyba że dodasz `inplace=True`).

In [ ]:
# Sortowanie rosnąco wg wieku
dataset.sort_values(by='Age')

In [ ]:
# Sortowanie malejąco wg ceny biletu (najdrożsi pasażerowie na górze)
dataset.sort_values(by='Fare', ascending=False)

### Sortowanie po kilku kolumnach naraz

Podajemy **listę kolumn** do `by` oraz **listę kierunków** do `ascending`. pandas sortuje najpierw wg pierwszej kolumny, a przy remisach — wg kolejnej.

In [ ]:
# Najpierw klasa rosnąco, a w obrębie tej samej klasy — wiek malejąco
dataset.sort_values(by=['Pclass', 'Age'], ascending=[True, False])

### Co z brakami danych (`NaN`) przy sortowaniu?

Kolumna `Age` ma braki. **Domyślnie pandas wrzuca wszystkie `NaN` na sam koniec** — niezależnie od tego, czy sortujemy rosnąco, czy malejąco. Zobaczmy to, sortując malejąco i podglądając ogon tabeli:

In [ ]:
# Domyślnie: NaN-y na końcu (widać je na dole, mimo sortowania malejąco)
dataset.sort_values(by='Age', ascending=False).tail(10)

Zachowaniem braków sterujemy parametrem **`na_position`**:

| Wartość | Gdzie trafią `NaN` |
|---------|---------------------|
| `'last'`  | na końcu (domyślnie) |
| `'first'` | na początku |

Poniżej `na_position='first'` — i nagle wszystkie braki wieku **wyskakują na samą górę**:

In [ ]:
# NaN-y wymuszone na początek tabeli
dataset.sort_values(by='Age', na_position='first')

In [ ]:
# Działa też przy sortowaniu po kilku kolumnach
dataset.sort_values(by=['Pclass', 'Age'], ascending=[True, False], na_position='first')

<a id="5"></a>
## 5. 🔎 Filtrowanie wierszy

Żeby wybrać tylko wiersze spełniające warunek, wstawiamy ten warunek w nawiasy kwadratowe:
`dataset[warunek]`.

Warunek to porównanie na kolumnie, np. `dataset.Age > 30` — zwraca kolumnę wartości `True`/`False`, a pandas zostawia tylko wiersze z `True`.

**Łączenie warunków** (uwaga — inne operatory niż zwykłe `and`/`or` w Pythonie!):

| Operator | Znaczenie | Warunek prawdziwy, gdy... |
|----------|-----------|----------------------------|
| `&`      | **i** (AND)  | spełnione **oba** warunki |
| `\|`     | **lub** (OR) | spełniony **co najmniej jeden** |
| `~`      | **negacja**  | warunek **nie** jest spełniony |

⚠️ Każdy warunek **musi być w osobnych nawiasach**: `dataset[(warunek1) & (warunek2)]` — bez nawiasów pandas zgłosi błąd.

In [ ]:
# Pojedynczy warunek — pasażerowie powyżej 30. roku życia
dataset[dataset.Age > 30]

In [ ]:
# Dwa warunki naraz (AND) — mężczyźni z 1. klasy
dataset[(dataset.Sex == 'male') & (dataset.Pclass == 1)]

In [ ]:
# Alternatywa (OR) — pasażerowie z 1. LUB 2. klasy
dataset[(dataset.Pclass == 1) | (dataset.Pclass == 2)]

<a id="6"></a>
## 6. 🛟 Kto przeżył?

`value_counts()` zlicza wystąpienia wartości. Z `normalize=True` dostaniemy **udziały procentowe**.

In [ ]:
dataset.Survived.value_counts()

In [ ]:
# Odsetek ocalałych vs. ofiar
dataset.Survived.value_counts(normalize=True)

In [ ]:
# Podział pasażerów wg klasy biletu
dataset.Pclass.value_counts()

<a id="7"></a>
## 7. 📊 Statystyki opisowe

`describe()` zwraca podsumowanie kolumn liczbowych: liczność, średnią, odchylenie, min/max i kwartyle.

In [ ]:
dataset.describe()

<a id="8"></a>
## 8. 🔄 Powrót z DataFrame do Pythona

Czasem chcemy wyjść z `pandas` z powrotem do **czystych obiektów Pythona** — np. żeby przekazać dane do API (JSON), zapisać do pliku albo iterować pętlą `for`.

Najczęściej używana jest metoda **`to_dict()`**, która ma kilka „kształtów" wyniku sterowanych argumentem `orient`:

| `orient` | Co zwraca |
|----------|-----------|
| `'dict'` (domyślnie) | słownik `{ kolumna: { indeks: wartość } }` |
| `'records'` | **lista słowników** — po jednym na wiersz (najczytelniejsze, jak JSON) |
| `'list'` | słownik `{ kolumna: [wartości...] }` |

Bierzemy tylko `head(3)`, żeby wynik był krótki.

In [ ]:
# Domyślnie: słownik {kolumna: {indeks: wartość}}
dataset.head(3).to_dict()

In [ ]:
# 'records' — lista słowników, po jednym na wiersz (format jak JSON)
dataset.head(3).to_dict('records')

In [ ]:
# Pojedynczą kolumnę najłatwiej zamienić na zwykłą listę Pythona
dataset['Sex'].head(3).tolist()

In [ ]:
# Sprawdźmy, że to już czyste typy Pythona
rekordy = dataset.head(3).to_dict('records')
print(type(rekordy))       # <class 'list'>
print(type(rekordy[0]))    # <class 'dict'>
rekordy[0]

<a id="9"></a>
## 9. 🎯 Zadania do samodzielnego rozwiązania

Każde zadanie odpowiada jednej sekcji z góry — rozwiąż je tymi samymi metodami, które właśnie poznałeś. Najpierw spróbuj sam, a podpowiedź i rozwiązanie odsłoń dopiero, gdy utkniesz.

---

### Zadanie 1 — *wybór kolumn* (sekcja 1)
Wyświetl tabelę zawężoną tylko do kolumn `Name`, `Sex` i `Age`.

<details><summary>💡 Podpowiedź</summary>

Podaj **listę** nazw kolumn w nawiasach kwadratowych: `dataset[[ ... ]]`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset[['Name', 'Sex', 'Age']]
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 1:


### Zadanie 2 — *pierwszy rzut oka* (sekcja 2)
Wyświetl **ostatnie 10** wierszy zbioru.

<details><summary>💡 Podpowiedź</summary>

Metoda `tail()` przyjmuje liczbę wierszy jako argument.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.tail(10)
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 2:


### Zadanie 3 — *typy i braki danych* (sekcja 3)
Sprawdź, **ile braków danych (`NaN`)** ma kolumna `Cabin`.

<details><summary>💡 Podpowiedź</summary>

Połącz `isna()` z `sum()` — ale tym razem na pojedynczej kolumnie: `dataset.Cabin`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.Cabin.isna().sum()
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 3:


### Zadanie 4 — *sortowanie* (sekcja 4)
Posortuj pasażerów **malejąco wg ceny biletu** (`Fare`) i wyświetl **5 najdroższych**.

<details><summary>💡 Podpowiedź</summary>

Użyj `sort_values(by=..., ascending=False)`, a na końcu dołóż `.head(5)`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.sort_values(by='Fare', ascending=False).head(5)
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 4:


### Zadanie 5 — *filtrowanie* (sekcja 5)
Wyświetl tylko **kobiety, które przeżyły** (`Sex == 'female'` **oraz** `Survived == 1`).

<details><summary>💡 Podpowiedź</summary>

Połącz dwa warunki operatorem `&` i pamiętaj o nawiasach: `dataset[(warunek1) & (warunek2)]`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset[(dataset.Sex == 'female') & (dataset.Survived == 1)]
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 5:


### Zadanie 6 — *zliczanie wartości* (sekcja 6)
Policz, **ilu pasażerów zaokrętowało się w każdym porcie** (`Embarked`).

<details><summary>💡 Podpowiedź</summary>

Metoda `value_counts()` na kolumnie `Embarked`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.Embarked.value_counts()
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 6:


### Zadanie 7 — *statystyki* (sekcja 7)
Oblicz **średni wiek** (`Age`) pasażerów.

<details><summary>💡 Podpowiedź</summary>

Pełne podsumowanie da `describe()`, ale samą średnią policzysz metodą `mean()` na kolumnie.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.Age.mean()
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 7:


### Zadanie 8 — *powrót do Pythona* (sekcja 8)
Zamień pierwsze **5 wierszy** zbioru na **listę słowników** (format jak JSON).

<details><summary>💡 Podpowiedź</summary>

Połącz `head(5)` z `to_dict('records')`.
</details>

<details><summary>✅ Rozwiązanie</summary>

```python
dataset.head(5).to_dict('records')
```
</details>

In [ ]:
# Twoje rozwiązanie zadania 8:
